# ViaAlerta — Demonstração do Modelo de Visão Computacional (YOLOv8)
**Global Solution FIAP 2026.2 — CitizenTech**

Este notebook demonstra a validação e o pipeline de inferência do modelo YOLOv8 treinado para identificação de buracos e avarias na malha asfáltica urbana.

In [1]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import json
from pathlib import Path

# 1. Carregamento dos Metadados do Dataset
metadata_path = Path('../data/metadata_dataset.json')
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

In [2]:
# 2. Carregamento do Modelo Pré-treinado YOLOv8
model_path = Path('../data/models/pothole_yolov8.pt')
model = YOLO(str(model_path))
print('Modelo carregado com sucesso!')
print('Classes suportadas:', model.names)

In [3]:
# 3. Inferência em Imagem Real com Buraco
sample_img_path = Path('../data/samples/buraco_sample_1.jpg')
results = model.predict(source=str(sample_img_path), conf=0.25)
res = results[0]

print(f'Total de buracos detectados: {len(res.boxes)}')
for i, box in enumerate(res.boxes):
    coords = [round(float(c), 1) for c in box.xyxy[0]]
    conf = round(float(box.conf[0]) * 100, 2)
    print(f' - Caixa #{i+1}: Coordenadas [xmin, ymin, xmax, ymax]={coords} | Confiança: {conf}%')

In [4]:
# 4. Cálculo do Índice de Severidade Visual e Explicabilidade
with Image.open(sample_img_path) as img:
    w, h = img.size

total_pixels = w * h
area_caixas = sum([(float(b.xyxy[0][2] - b.xyxy[0][0]) * float(b.xyxy[0][3] - b.xyxy[0][1])) for b in res.boxes])
area_pct = round((area_caixas / total_pixels) * 100, 2)

print(f'Área visual relativa ocupada: {area_pct}% da fotografia')
if area_pct >= 10.0 or len(res.boxes) >= 3:
    severidade = 'Severa'
elif area_pct >= 3.5 or len(res.boxes) >= 2:
    severidade = 'Moderada'
else:
    severidade = 'Leve'
print(f'Classificação visual atribuída: {severidade}')